# GéoMarketing IDF : J11 — Cartographie et contrôles géographiques en Île-de-France

## Objectifs
- Associer le profil communal aux contours géographiques.
- Contrôler les coordonnées des observations détaillées.
- Comparer leur position à la commune déclarée.
- Produire des diagnostics et des cartes interactives.

## Principes
- Conserver les codes communaux déclarés.
- Signaler les écarts sans réaffectation automatique.
- Conserver les observations sans coordonnées dans les comptages.
- Documenter la source et la date de récupération des contours.

## Limites
- Les contours utilisés constituent une référence cartographique.
- Une position dans le bon contour ne valide pas l'adresse exacte.
- Une distance géométrique n'est pas une distance de marche.
- Les cartes ne constituent pas encore un score d'implantation.

In [1]:
%pip install geopandas folium

In [2]:
#Import librairies

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.parse import urlencode

import hashlib
import html
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import geopandas as gpd
import folium

from folium.plugins import MarkerCluster
from branca.colormap import linear
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("Pandas :", pd.__version__)
print("GeoPandas :", gpd.__version__)
print("Folium :", folium.__version__)

Pandas : 2.2.2
GeoPandas : 1.0.1
Folium : 0.20.0


In [3]:
#Définir les chemins

RACINE = Path(
    r"C:\Users\almou\OneDrive\GeoMarketing_IDF"
)

DOSSIER_RAW_GEO = RACINE / "data" / "raw" / "geographie"
DOSSIER_INTERIM = RACINE / "data" / "interim" / "j11"
DOSSIER_PROCESSED = RACINE / "data" / "processed"
DOSSIER_GEO = DOSSIER_PROCESSED / "j11"

DOSSIER_CARTES = RACINE / "outputs" / "cartes" / "j11"

FICHIER_PROFIL = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10g.csv"
)

# Si J10h est déjà terminé, remplace la ligne précédente par :
# FICHIER_PROFIL = (
#     DOSSIER_PROCESSED / "profil_communes_idf_j10h.csv"
# )

FICHIER_CONTOURS = (
    DOSSIER_RAW_GEO / "communes_idf_api.geojson"
)

FICHIER_METADATA_CONTOURS = (
    DOSSIER_RAW_GEO / "communes_idf_api_metadata.json"
)

FICHIER_PROFIL_J11 = (
    DOSSIER_PROCESSED / "profil_communes_idf_j11.csv"
)

for dossier in [
    DOSSIER_RAW_GEO,
    DOSSIER_INTERIM,
    DOSSIER_GEO,
    DOSSIER_CARTES,
]:
    dossier.mkdir(parents=True, exist_ok=True)

if not FICHIER_PROFIL.is_file():
    raise FileNotFoundError(FICHIER_PROFIL)

print("Profil d'entrée :", FICHIER_PROFIL)
print("Cartes :", DOSSIER_CARTES)

Profil d'entrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10g.csv
Cartes : C:\Users\almou\OneDrive\GeoMarketing_IDF\outputs\cartes\j11


In [4]:
#Fonctions utiles

def normaliser_nom_colonne(nom):
    nom = unicodedata.normalize(
        "NFKD", str(nom).strip().upper()
    )
    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^A-Z0-9]+", "_", nom).strip("_")


def nettoyer_texte(serie):
    resultat = serie.astype("string").str.strip()

    manquants = {
        "", "NA", "NAN", "NONE", "NULL",
        "<NA>", "_U", "_Z", "[ND]",
    }

    return resultat.mask(
        resultat.str.upper().isin(manquants)
    )


def normaliser_code_commune(serie):
    resultat = nettoyer_texte(serie).str.upper()
    resultat = resultat.str.replace(r"\.0$", "", regex=True)

    masque = resultat.str.fullmatch(r"\d{1,5}", na=False)

    return resultat.where(
        ~masque,
        resultat.str.zfill(5),
    )


def convertir_nombre(serie):
    return pd.to_numeric(
        nettoyer_texte(serie)
        .str.replace("\u202f", "", regex=False)
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def sauvegarder_csv(table, chemin):
    table.to_csv(
        chemin,
        index=False,
        encoding="utf-8-sig",
    )


def sha256_fichier(chemin):
    empreinte = hashlib.sha256()

    with open(chemin, "rb") as flux:
        for bloc in iter(lambda: flux.read(1024 * 1024), b""):
            empreinte.update(bloc)

    return empreinte.hexdigest()

In [5]:
#Télécharger les contours communaux

DEPARTEMENTS_IDF = [
    "75", "77", "78", "91", "92", "93", "94", "95"
]

# False : réutiliser la copie déjà téléchargée.
# True : récupérer une nouvelle version des contours.
ACTUALISER_CONTOURS = False

if ACTUALISER_CONTOURS or not FICHIER_CONTOURS.exists():
    toutes_features = []
    urls_sources = []

    parametres = urlencode({
        "fields": "code,nom,codeDepartement",
        "format": "geojson",
        "geometry": "contour",
    })

    for departement in DEPARTEMENTS_IDF:
        url = (
            f"https://geo.api.gouv.fr/departements/"
            f"{departement}/communes?{parametres}"
        )

        requete = Request(
            url,
            headers={"User-Agent": "GeoMarketing_IDF"},
        )

        with urlopen(requete, timeout=45) as reponse:
            contenu = json.loads(
                reponse.read().decode("utf-8")
            )

        if contenu.get("type") != "FeatureCollection":
            raise ValueError(
                f"Réponse GeoJSON inattendue : {departement}"
            )

        features = contenu.get("features", [])

        if not features:
            raise ValueError(
                f"Aucun contour reçu pour le département {departement}"
            )

        toutes_features.extend(features)
        urls_sources.append(url)

        print(departement, ":", len(features), "communes")

    geojson = {
        "type": "FeatureCollection",
        "features": toutes_features,
    }

    # Enregistrement seulement après récupération des huit départements.
    with open(FICHIER_CONTOURS, "w", encoding="utf-8") as flux:
        json.dump(geojson, flux, ensure_ascii=False)

    metadata_contours = {
        "source": "API_DECOUPAGE_ADMINISTRATIF",
        "urls": urls_sources,
        "date_recuperation_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "millesime_annuel": None,
        "sha256": sha256_fichier(FICHIER_CONTOURS),
    }

    with open(
        FICHIER_METADATA_CONTOURS,
        "w",
        encoding="utf-8",
    ) as flux:
        json.dump(
            metadata_contours,
            flux,
            ensure_ascii=False,
            indent=2,
        )

else:
    print("Contours locaux réutilisés :", FICHIER_CONTOURS)

75 : 1 communes
77 : 507 communes
78 : 259 communes
91 : 194 communes
92 : 36 communes
93 : 39 communes
94 : 47 communes
95 : 183 communes


In [6]:
#Charger le profil et les contours

profil = pd.read_csv(
    FICHIER_PROFIL,
    dtype="string",
    encoding="utf-8-sig",
)

profil.columns = [
    normaliser_nom_colonne(c) for c in profil.columns
]

if profil.columns.duplicated().any():
    raise ValueError("Colonnes dupliquées dans le profil.")

if "CODGEO" not in profil.columns:
    raise ValueError("CODGEO absent du profil.")

profil["CODGEO"] = normaliser_code_commune(profil["CODGEO"])

if not profil["CODGEO"].str.fullmatch(r"\d{5}", na=False).all():
    raise ValueError("Codes communaux invalides.")

if not profil["CODGEO"].is_unique:
    raise ValueError("Codes communaux dupliqués.")

communes = gpd.read_file(FICHIER_CONTOURS)

if not {"code", "nom"}.issubset(communes.columns):
    raise ValueError(
        f"Colonnes des contours inattendues : {communes.columns.tolist()}"
    )

communes = communes[
    ["code", "nom", "geometry"]
].rename(columns={
    "code": "CODGEO",
    "nom": "NOM_COMMUNE_CONTOUR",
})

communes["CODGEO"] = normaliser_code_commune(
    communes["CODGEO"]
)

if not communes["CODGEO"].is_unique:
    raise ValueError("Plusieurs contours existent pour un même code.")

if not communes["CODGEO"].str.fullmatch(r"\d{5}", na=False).all():
    raise ValueError("Codes invalides dans les contours.")

if communes.crs is None:
    raise ValueError("Le système de coordonnées des contours est absent.")

communes = communes.to_crs("EPSG:4326")

print("Communes du profil :", len(profil))
print("Contours :", len(communes))
print("Système de coordonnées :", communes.crs)

Communes du profil : 1266
Contours : 1266
Système de coordonnées : EPSG:4326


In [7]:
#Vérifier les géométries

geometries_problematiques = (
    communes.geometry.isna()
    | communes.geometry.is_empty
    | ~communes.geometry.is_valid
    | ~communes.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    )
)

if geometries_problematiques.any():
    display(
        communes.loc[
            geometries_problematiques,
            ["CODGEO", "NOM_COMMUNE_CONTOUR"],
        ]
    )

    raise ValueError(
        "Des contours sont absents, invalides ou non polygonaux."
    )

codes_profil = set(profil["CODGEO"])
codes_contours = set(communes["CODGEO"])

codes_sans_contour = sorted(codes_profil - codes_contours)
codes_sans_profil = sorted(codes_contours - codes_profil)

sauvegarder_csv(
    pd.DataFrame({"CODGEO": codes_sans_contour}),
    DOSSIER_INTERIM / "communes_sans_contour.csv",
)

sauvegarder_csv(
    pd.DataFrame({"CODGEO": codes_sans_profil}),
    DOSSIER_INTERIM / "contours_sans_profil.csv",
)

print("Communes du profil sans contour :", len(codes_sans_contour))
print("Contours sans profil :", len(codes_sans_profil))

if codes_sans_contour:
    raise ValueError(
        "Des communes du profil n'ont pas de contour. "
        "Vérifier les codes et la compatibilité des géographies."
    )

Communes du profil sans contour : 0
Contours sans profil : 0


In [8]:
#Créer la version cartographique du profil

communes_metres = communes.to_crs("EPSG:2154")

surfaces = pd.DataFrame({
    "CODGEO": communes_metres["CODGEO"],
    "SUPERFICIE_CONTOUR_KM2": (
        communes_metres.geometry.area / 1_000_000
    ),
})

if "SUPERFICIE_CONTOUR_KM2" in profil.columns:
    raise ValueError("La superficie du contour existe déjà dans le profil.")

profil_j11 = profil.merge(
    surfaces,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

assert profil_j11["SUPERFICIE_CONTOUR_KM2"].gt(0).all()

profil_geo = communes.merge(
    profil_j11,
    on="CODGEO",
    how="inner",
    validate="one_to_one",
)

assert len(profil_geo) == len(profil)

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j11[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

print("Profil cartographique :", len(profil_geo), "communes")

Profil cartographique : 1266 communes


In [10]:
#Définir les tables détaillées à controler

TABLES_DETAIL = {
    "RESTAURANTS":
        DOSSIER_PROCESSED / "restaurants_idf_detail_j9b.csv",

    "COMMERCES_ALIMENTAIRES":
        DOSSIER_PROCESSED / "commerces_alimentaires_idf_j10b.csv",

    "SERVICES_PARTICULIERS":
        DOSSIER_PROCESSED / "services_particuliers_idf_j10c.csv",

    "SANTE_ACTION_SOCIALE":
        DOSSIER_PROCESSED / "sante_action_sociale_idf_j10d.csv",

    "SPORTS_LOISIRS_CULTURE":
        DOSSIER_PROCESSED / "sports_loisirs_culture_idf_j10e.csv",

    "EDUCATION":
        DOSSIER_PROCESSED / "education_idf_j10f.csv",

    "TOURISME":
        DOSSIER_PROCESSED / "tourisme_hebergements_idf_j10g.csv",
}

for theme, fichier in TABLES_DETAIL.items():
    if not fichier.is_file():
        raise FileNotFoundError(f"{theme} : {fichier}")

display(
    pd.DataFrame([
        {"THEME": theme, "FICHIER": str(fichier)}
        for theme, fichier in TABLES_DETAIL.items()
    ])
)

,THEME,FICHIER
0,RESTAURANTS,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\restaurants_idf_detail_j9b.csv
1,COMMERCES_ALIMENTAIRES,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\commerces_alimentaires_idf_j10b.csv
2,SERVICES_PARTICULIERS,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\services_particuliers_idf_j10c.csv
3,SANTE_ACTION_SOCIALE,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\sante_action_sociale_idf_j10d.csv
4,SPORTS_LOISIRS_CULTURE,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\sports_loisirs_culture_idf_j10e.csv
5,EDUCATION,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\education_idf_j10f.csv
6,TOURISME,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\tourisme_hebergements_idf_j10g.csv


In [11]:
#Définir les statuts géographiques

TOLERANCE_METRES = 20.0

contours_controle = communes_metres[
    ["CODGEO", "geometry"]
].copy()

contours_controle = contours_controle.rename(
    columns={"CODGEO": "CODGEO_CONTOUR"}
)

geometries_par_code = communes_metres.set_index(
    "CODGEO"
).geometry

In [12]:
#Fonction de controle géographiques

def controler_table_geographique(fichier, theme):
    table = pd.read_csv(
        fichier,
        dtype="string",
        encoding="utf-8-sig",
    )

    table.columns = [
        normaliser_nom_colonne(c) for c in table.columns
    ]

    if table.columns.duplicated().any():
        raise ValueError(f"{theme} : colonnes dupliquées.")

    if "CODGEO" not in table.columns:
        raise ValueError(f"{theme} : CODGEO absent.")

    table["CODGEO"] = normaliser_code_commune(table["CODGEO"])

    if "ID_OBSERVATION" not in table.columns:
        raise ValueError(f"{theme} : ID_OBSERVATION absent.")

    if table["ID_OBSERVATION"].isna().any():
        raise ValueError(f"{theme} : identifiants manquants.")

    if not table["ID_OBSERVATION"].is_unique:
        raise ValueError(f"{theme} : identifiants dupliqués.")

    def trouver_colonne(candidats):
        return next(
            (c for c in candidats if c in table.columns),
            None,
        )

    colonne_lat = trouver_colonne(["LATITUDE_NUM", "LATITUDE"])
    colonne_lon = trouver_colonne(["LONGITUDE_NUM", "LONGITUDE"])

    if colonne_lat is None or colonne_lon is None:
        raise ValueError(f"{theme} : colonnes de coordonnées absentes.")

    table["LATITUDE_CONTROLE"] = convertir_nombre(table[colonne_lat])
    table["LONGITUDE_CONTROLE"] = convertir_nombre(table[colonne_lon])

    # Identifiant interne de ligne pour les jointures spatiales.
    table["LIGNE_CONTROLE"] = np.arange(len(table))

    table["STATUT_GEO_J11"] = (
        "COORDONNEES_ABSENTES_OU_NON_NUMERIQUES"
    )

    presentes = (
        table["LATITUDE_CONTROLE"].notna()
        & table["LONGITUDE_CONTROLE"].notna()
    )

    # Rectangle large de contrôle pour la métropole.
    # Les coordonnées extérieures ne sont pas projetées automatiquement.
    projetables = (
        presentes
        & table["LATITUDE_CONTROLE"].between(41.0, 52.0)
        & table["LONGITUDE_CONTROLE"].between(-6.0, 10.0)
    )

    table.loc[
        presentes & ~projetables,
        "STATUT_GEO_J11",
    ] = "COORDONNEES_HORS_ZONE_CONTROLE"

    table["DISTANCE_COMMUNE_DECLAREE_M"] = np.nan
    table["NB_COMMUNES_INTERSECTEES"] = 0
    table["CODES_COMMUNES_INTERSECTEES"] = ""

    points = gpd.GeoDataFrame(
        table.loc[
            projetables,
            ["LIGNE_CONTROLE", "ID_OBSERVATION", "CODGEO"],
        ].copy(),
        geometry=gpd.points_from_xy(
            table.loc[projetables, "LONGITUDE_CONTROLE"],
            table.loc[projetables, "LATITUDE_CONTROLE"],
        ),
        crs="EPSG:4326",
    ).to_crs("EPSG:2154")

    if not points.empty:
        intersections = gpd.sjoin(
            points,
            contours_controle,
            how="left",
            predicate="intersects",
        )

        def assembler_codes(serie):
            return ", ".join(
                sorted(set(serie.dropna().astype(str)))
            )

        codes_intersectes = (
            intersections.groupby("LIGNE_CONTROLE")["CODGEO_CONTOUR"]
            .agg(assembler_codes)
        )

        nombres_intersections = (
            intersections.groupby("LIGNE_CONTROLE")["CODGEO_CONTOUR"]
            .nunique()
        )

        table.loc[
            projetables, "CODES_COMMUNES_INTERSECTEES"
        ] = (
            table.loc[projetables, "LIGNE_CONTROLE"]
            .map(codes_intersectes)
            .fillna("")
        )

        table.loc[
            projetables, "NB_COMMUNES_INTERSECTEES"
        ] = (
            table.loc[projetables, "LIGNE_CONTROLE"]
            .map(nombres_intersections)
            .fillna(0)
            .astype(int)
        )

        # Distance au polygone de la commune déclarée.
        for code, groupe_points in points.groupby(
            "CODGEO", dropna=False
        ):
            indices = groupe_points.index

            if pd.isna(code) or code not in geometries_par_code.index:
                table.loc[
                    indices, "STATUT_GEO_J11"
                ] = "COMMUNE_SANS_CONTOUR"
                continue

            polygone = geometries_par_code.loc[code]

            distances = groupe_points.geometry.distance(polygone)
            dans_commune = groupe_points.geometry.intersects(polygone)

            table.loc[
                indices, "DISTANCE_COMMUNE_DECLAREE_M"
            ] = distances

            table.loc[
                indices, "STATUT_GEO_J11"
            ] = "ECART_COMMUNE_A_EXAMINER"

            proches = ~dans_commune & distances.le(TOLERANCE_METRES)

            table.loc[
                proches.index[proches], "STATUT_GEO_J11"
            ] = "PROCHE_COMMUNE_DECLAREE_A_VERIFIER"

            table.loc[
                dans_commune.index[dans_commune], "STATUT_GEO_J11"
            ] = "DANS_COMMUNE_DECLAREE"

        points["STATUT_GEO_J11"] = table.loc[
            points.index, "STATUT_GEO_J11"
        ]

        points["NB_COMMUNES_INTERSECTEES"] = table.loc[
            points.index, "NB_COMMUNES_INTERSECTEES"
        ]

    table["THEME_CONTROLE"] = theme

    assert len(table) == table["ID_OBSERVATION"].nunique()

    return table, points

In [13]:
#Bilan géographique

bilans = []
manifestes_tables = []

STATUTS_GEO = [
    "DANS_COMMUNE_DECLAREE",
    "PROCHE_COMMUNE_DECLAREE_A_VERIFIER",
    "ECART_COMMUNE_A_EXAMINER",
    "COMMUNE_SANS_CONTOUR",
    "COORDONNEES_ABSENTES_OU_NON_NUMERIQUES",
    "COORDONNEES_HORS_ZONE_CONTROLE",
]

for theme, fichier in TABLES_DETAIL.items():
    print("Contrôle :", theme)

    table_controle, points = controler_table_geographique(
        fichier, theme
    )

    fichier_controle = (
        DOSSIER_INTERIM / f"{theme.lower()}_controle_geo.csv"
    )

    sauvegarder_csv(table_controle, fichier_controle)

    anomalies = table_controle.loc[
        table_controle["STATUT_GEO_J11"].ne("DANS_COMMUNE_DECLAREE")
        | table_controle["NB_COMMUNES_INTERSECTEES"].gt(1)
    ]

    sauvegarder_csv(
        anomalies,
        DOSSIER_INTERIM / f"{theme.lower()}_a_examiner.csv",
    )

    fichier_points = DOSSIER_GEO / f"{theme.lower()}_points.gpkg"

    # Remplacer uniquement notre export J11 de ce thème.
    if fichier_points.exists():
        fichier_points.unlink()

    if not points.empty:
        points.to_file(
            fichier_points,
            layer="points",
            driver="GPKG",
            index=False,
        )

    effectifs = (
        table_controle["STATUT_GEO_J11"]
        .value_counts()
        .reindex(STATUTS_GEO, fill_value=0)
    )

    for statut, nombre in effectifs.items():
        bilans.append({
            "THEME": theme,
            "STATUT": statut,
            "NB_OBSERVATIONS": int(nombre),
            "PART_PCT": nombre / len(table_controle) * 100,
        })

    manifestes_tables.append({
        "THEME": theme,
        "FICHIER_SOURCE": str(fichier),
        "SHA256_SOURCE": sha256_fichier(fichier),
        "NB_OBSERVATIONS": len(table_controle),
        "NB_POINTS_PROJETABLES": len(points),
        "NB_INTERSECTIONS_MULTIPLES": int(
            table_controle["NB_COMMUNES_INTERSECTEES"].gt(1).sum()
        ),
    })

    del table_controle, points, anomalies

bilan_geo = pd.DataFrame(bilans)
manifeste_tables = pd.DataFrame(manifestes_tables)

sauvegarder_csv(
    bilan_geo,
    DOSSIER_INTERIM / "bilan_qualite_geographique.csv",
)

sauvegarder_csv(
    manifeste_tables,
    DOSSIER_INTERIM / "manifeste_tables_controlees.csv",
)

display(bilan_geo)

Contrôle : RESTAURANTS
Contrôle : COMMERCES_ALIMENTAIRES
Contrôle : SERVICES_PARTICULIERS
Contrôle : SANTE_ACTION_SOCIALE
Contrôle : SPORTS_LOISIRS_CULTURE
Contrôle : EDUCATION
Contrôle : TOURISME


,THEME,STATUT,NB_OBSERVATIONS,PART_PCT
0,RESTAURANTS,DANS_COMMUNE_DECLAREE,47828,97.272672
1,RESTAURANTS,PROCHE_COMMUNE_DECLAREE_A_VERIFIER,15,0.030507
2,RESTAURANTS,ECART_COMMUNE_A_EXAMINER,13,0.026439
3,RESTAURANTS,COMMUNE_SANS_CONTOUR,0,0.000000
4,RESTAURANTS,COORDONNEES_ABSENTES_OU_NON_NUMERIQUES,1313,2.670382
5,RESTAURANTS,COORDONNEES_HORS_ZONE_CONTROLE,0,0.000000
6,COMMERCES_ALIMENTAIRES,DANS_COMMUNE_DECLAREE,26865,98.144157
7,COMMERCES_ALIMENTAIRES,PROCHE_COMMUNE_DECLAREE_A_VERIFIER,5,0.018266
8,COMMERCES_ALIMENTAIRES,ECART_COMMUNE_A_EXAMINER,7,0.025573
9,COMMERCES_ALIMENTAIRES,COMMUNE_SANS_CONTOUR,0,0.000000


In [14]:
#Lire les résultats du controle

vue_bilan = bilan_geo.pivot(
    index="THEME",
    columns="STATUT",
    values="NB_OBSERVATIONS",
).fillna(0).astype(int)

display(vue_bilan)

print("Cas à examiner en priorité :")

display(
    bilan_geo.loc[
        bilan_geo["STATUT"].isin([
            "ECART_COMMUNE_A_EXAMINER",
            "COMMUNE_SANS_CONTOUR",
            "COORDONNEES_HORS_ZONE_CONTROLE",
        ])
    ]
)

STATUT,COMMUNE_SANS_CONTOUR,COORDONNEES_ABSENTES_OU_NON_NUMERIQUES,COORDONNEES_HORS_ZONE_CONTROLE,DANS_COMMUNE_DECLAREE,ECART_COMMUNE_A_EXAMINER,PROCHE_COMMUNE_DECLAREE_A_VERIFIER
THEME,,,,,,
COMMERCES_ALIMENTAIRES,0,496,0,26865,7,5
EDUCATION,0,5,0,12707,5,9
RESTAURANTS,0,1313,0,47828,13,15
SANTE_ACTION_SOCIALE,0,45,0,112917,42,69
SERVICES_PARTICULIERS,0,13115,0,171558,34,53
SPORTS_LOISIRS_CULTURE,0,124,0,20346,2,6
TOURISME,0,1043,0,8070,2,5


Cas à examiner en priorité :


,THEME,STATUT,NB_OBSERVATIONS,PART_PCT
2,RESTAURANTS,ECART_COMMUNE_A_EXAMINER,13,0.026439
3,RESTAURANTS,COMMUNE_SANS_CONTOUR,0,0.000000
5,RESTAURANTS,COORDONNEES_HORS_ZONE_CONTROLE,0,0.000000
8,COMMERCES_ALIMENTAIRES,ECART_COMMUNE_A_EXAMINER,7,0.025573
9,COMMERCES_ALIMENTAIRES,COMMUNE_SANS_CONTOUR,0,0.000000
11,COMMERCES_ALIMENTAIRES,COORDONNEES_HORS_ZONE_CONTROLE,0,0.000000
14,SERVICES_PARTICULIERS,ECART_COMMUNE_A_EXAMINER,34,0.018402
15,SERVICES_PARTICULIERS,COMMUNE_SANS_CONTOUR,0,0.000000
17,SERVICES_PARTICULIERS,COORDONNEES_HORS_ZONE_CONTROLE,0,0.000000
20,SANTE_ACTION_SOCIALE,ECART_COMMUNE_A_EXAMINER,42,0.037144


In [15]:
#Enregistrer les communes cartographiques

FICHIER_COMMUNES_GPKG = (
    DOSSIER_GEO / "profil_communes_idf_j11.gpkg"
)

FICHIER_COMMUNES_GEOJSON = (
    DOSSIER_GEO / "communes_idf_j11.geojson"
)

if FICHIER_COMMUNES_GPKG.exists():
    FICHIER_COMMUNES_GPKG.unlink()

profil_geo.to_crs("EPSG:2154").to_file(
    FICHIER_COMMUNES_GPKG,
    layer="communes",
    driver="GPKG",
    index=False,
)

profil_geo[
    ["CODGEO", "NOM_COMMUNE_CONTOUR", "geometry"]
].to_file(
    FICHIER_COMMUNES_GEOJSON,
    driver="GeoJSON",
    index=False,
)

sauvegarder_csv(profil_j11, FICHIER_PROFIL_J11)

print("Profil CSV :", FICHIER_PROFIL_J11)
print("Profil géographique :", FICHIER_COMMUNES_GPKG)

Profil CSV : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j11.csv
Profil géographique : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j11\profil_communes_idf_j11.gpkg


In [25]:
#Fonction de carte communale

def creer_carte_communes(colonne, titre, nom_fichier):
    if colonne not in profil_geo.columns:
        raise ValueError(f"Indicateur absent : {colonne}")

    donnees = profil_geo[
        ["CODGEO", "NOM_COMMUNE_CONTOUR", colonne, "geometry"]
    ].copy()

    valeurs = convertir_nombre(donnees[colonne])
    valeurs = valeurs.where(
        np.isfinite(valeurs.fillna(np.nan).astype(float))
    )

    donnees["VALEUR"] = valeurs.astype(float)
    donnees = donnees.drop(columns=[colonne])

    # Simplification réservée à la carte.
    donnees = donnees.to_crs("EPSG:2154")
    donnees.geometry = donnees.geometry.simplify(
        20, preserve_topology=True
    )
    donnees = donnees.to_crs("EPSG:4326")

    disponibles = donnees["VALEUR"].dropna()

    if disponibles.empty:
        raise ValueError(f"Aucune valeur exploitable pour {colonne}")

    minimum = float(disponibles.min())
    maximum = float(disponibles.max())

    if minimum == maximum:
        maximum = minimum + 1

    couleurs = linear.YlOrRd_09.scale(minimum, maximum)
    couleurs.caption = titre

    carte = folium.Map(
        location=[48.85, 2.35],
        zoom_start=9,
        tiles="OpenStreetMap",
    )

    def style(feature):
        valeur = feature["properties"]["VALEUR"]

        return {
            "fillColor": (
                "#bdbdbd" if valeur is None else couleurs(valeur)
            ),
            "fillOpacity": 0.75,
            "color": "#ffffff",
            "weight": 0.5,
        }

    folium.GeoJson(
        data=json.loads(donnees.to_json(na="null")),
        name=titre,
        style_function=style,
        tooltip=folium.GeoJsonTooltip(
            fields=["NOM_COMMUNE_CONTOUR", "CODGEO", "VALEUR"],
            aliases=["Commune", "Code", titre],
            localize=True,
        ),
    ).add_to(carte)

    couleurs.add_to(carte)

    folium.Element(
        """
        <div style="position:fixed;bottom:25px;left:15px;
        background:white;padding:8px;z-index:9999;">
        Gris : valeur manquante
        </div>
        """
    ).add_to(carte.get_root().html)

    xmin, ymin, xmax, ymax = donnees.total_bounds
    carte.fit_bounds([[ymin, xmin], [ymax, xmax]])

    chemin = DOSSIER_CARTES / nom_fichier
    carte.save(str(chemin))

    return carte, chemin

In [26]:
#Créer trois premieres cartes

CARTES_A_CREER = [
    (
        "NIVEAU_VIE_MEDIAN_2023",
        "Niveau de vie médian 2023 — euros annuels par UC",
        "01_niveau_vie_median.html",
    ),
    (
        "DENSITE_OBS_ALIMENTAIRES_B2_10000_HAB",
        "Observations alimentaires B2 pour 10 000 habitants",
        "02_densite_commerces_alimentaires.html",
    ),
    (
        "DENSITE_OBS_CULTURE_10000_HAB",
        "Observations culturelles pour 10 000 habitants",
        "03_densite_culture.html",
    ),
]

cartes_creees = []

for colonne, titre, nom_fichier in CARTES_A_CREER:
    if colonne not in profil_geo.columns:
        print("Carte à configurer : indicateur absent", colonne)
        continue

    carte, chemin = creer_carte_communes(
        colonne, titre, nom_fichier
    )

    cartes_creees.append({
        "INDICATEUR": colonne,
        "TITRE": titre,
        "FICHIER": str(chemin),
    })

    print("Carte créée :", chemin)

    if colonne == "DENSITE_OBS_ALIMENTAIRES_B2_10000_HAB":
        display(carte)

Carte créée : C:\Users\almou\OneDrive\GeoMarketing_IDF\outputs\cartes\j11\01_niveau_vie_median.html
Carte créée : C:\Users\almou\OneDrive\GeoMarketing_IDF\outputs\cartes\j11\02_densite_commerces_alimentaires.html


Carte créée : C:\Users\almou\OneDrive\GeoMarketing_IDF\outputs\cartes\j11\03_densite_culture.html


In [27]:
carte_revenus, chemin_revenus = creer_carte_communes(
    "NIVEAU_VIE_MEDIAN_2023",
    "Niveau de vie médian 2023 — euros annuels par UC",
    "01_niveau_vie_median.html",
)

display(carte_revenus)

In [29]:
#Carte locale des restaurants de Cergy et Pontoise

restaurants_controles = pd.read_csv(
    DOSSIER_INTERIM / "restaurants_controle_geo.csv",
    dtype="string",
    encoding="utf-8-sig",
)

restaurants_controles["LATITUDE_CONTROLE"] = convertir_nombre(
    restaurants_controles["LATITUDE_CONTROLE"]
)

restaurants_controles["LONGITUDE_CONTROLE"] = convertir_nombre(
    restaurants_controles["LONGITUDE_CONTROLE"]
)

selection_locale = restaurants_controles.loc[
    restaurants_controles["CODGEO"].isin(["95127", "95500"])
    & restaurants_controles["LATITUDE_CONTROLE"].between(41, 52)
    & restaurants_controles["LONGITUDE_CONTROLE"].between(-6, 10)
].copy()

carte_locale = folium.Map(
    location=[49.04, 2.06],
    zoom_start=13,
    tiles="OpenStreetMap",
)

contours_locaux = communes.loc[
    communes["CODGEO"].isin(["95127", "95500"])
]

folium.GeoJson(
    json.loads(contours_locaux.to_json()),
    style_function=lambda feature: {
        "fillOpacity": 0.05,
        "color": "#333333",
        "weight": 2,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NOM_COMMUNE_CONTOUR"],
        aliases=["Commune"],
    ),
).add_to(carte_locale)

grappe = MarkerCluster(
    name="Restaurants — observations BPE"
).add_to(carte_locale)

for _, ligne in selection_locale.iterrows():
    statut = ligne["STATUT_GEO_J11"]

    if statut == "DANS_COMMUNE_DECLAREE":
        couleur = "green"
    elif statut == "PROCHE_COMMUNE_DECLAREE_A_VERIFIER":
        couleur = "orange"
    else:
        couleur = "red"

    nom = ligne.get("NOM_AFFICHAGE")
    if pd.isna(nom) or not str(nom).strip():
        nom = "Nom non disponible"

    popup = (
        f"<b>{html.escape(str(nom))}</b><br>"
        f"Commune déclarée : {html.escape(str(ligne['CODGEO']))}<br>"
        f"Contrôle : {html.escape(str(statut))}"
    )

    folium.Marker(
        location=[
            float(ligne["LATITUDE_CONTROLE"]),
            float(ligne["LONGITUDE_CONTROLE"]),
        ],
        popup=popup,
        icon=folium.Icon(color=couleur, icon="info-sign"),
    ).add_to(grappe)

folium.LayerControl().add_to(carte_locale)

FICHIER_CARTE_LOCALE = (
    DOSSIER_CARTES / "04_restaurants_cergy_pontoise_controle.html"
)

carte_locale.save(str(FICHIER_CARTE_LOCALE))

print("Restaurants représentés :", len(selection_locale))
display(carte_locale)

Restaurants représentés : 336


In [30]:
#Dociùenter les anomalies examinées

FICHIER_REVUE = DOSSIER_INTERIM / "revue_anomalies_geographiques.csv"

colonnes_revue = [
    "THEME",
    "ID_OBSERVATION",
    "STATUT_GEO_INITIAL",
    "RESULTAT_VERIFICATION",
    "SOURCE_VERIFICATION",
    "DATE_VERIFICATION",
    "LATITUDE_CORRIGEE",
    "LONGITUDE_CORRIGEE",
    "CODGEO_CORRIGE",
    "COMMENTAIRE",
]

if not FICHIER_REVUE.exists():
    sauvegarder_csv(
        pd.DataFrame(columns=colonnes_revue),
        FICHIER_REVUE,
    )

print("Fichier de revue :", FICHIER_REVUE)

Fichier de revue : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j11\revue_anomalies_geographiques.csv


## Transports — préparation des analyses locales

Le J9 fournit les indicateurs communaux et les données IDFM.

Avant les analyses de proximité :
- vérifier les identifiants des arrêts et lieux de transport ;
- distinguer arrêt, station, gare et accès ;
- contrôler les coordonnées des accès utilisés ;
- conserver les modes de transport ;
- documenter les points représentatifs ou regroupés.

Les distances à vol d'oiseau seront distinguées des distances piétonnes.

In [31]:
#Controles finaux et métadonnées

assert len(profil_j11) == len(profil)
assert len(profil_geo) == len(profil)

assert profil_j11["CODGEO"].is_unique
assert profil_geo["CODGEO"].is_unique

assert profil_geo.geometry.notna().all()
assert profil_geo.geometry.is_valid.all()

assert profil_j11["SUPERFICIE_CONTOUR_KM2"].gt(0).all()

for theme in TABLES_DETAIL:
    total_bilan = bilan_geo.loc[
        bilan_geo["THEME"].eq(theme),
        "NB_OBSERVATIONS",
    ].sum()

    total_source = manifeste_tables.loc[
        manifeste_tables["THEME"].eq(theme),
        "NB_OBSERVATIONS",
    ].iloc[0]

    assert total_bilan == total_source

pd.testing.assert_frame_equal(
    profil.sort_values("CODGEO").reset_index(drop=True),
    profil_j11[profil.columns]
    .sort_values("CODGEO")
    .reset_index(drop=True),
)

sauvegarder_csv(
    pd.DataFrame(cartes_creees),
    DOSSIER_INTERIM / "cartes_creees.csv",
)

metadata_contours = {}

if FICHIER_METADATA_CONTOURS.is_file():
    with open(
        FICHIER_METADATA_CONTOURS,
        "r",
        encoding="utf-8",
    ) as flux:
        metadata_contours = json.load(flux)

metadonnees = {
    "notebook": "11_cartographie_controles_geographiques_idf.ipynb",
    "date_traitement_utc": datetime.now(timezone.utc).isoformat(),
    "profil_entree": str(FICHIER_PROFIL),
    "sha256_profil_entree": sha256_fichier(FICHIER_PROFIL),
    "profil_sortie": str(FICHIER_PROFIL_J11),
    "fichier_contours": str(FICHIER_CONTOURS),
    "sha256_contours": sha256_fichier(FICHIER_CONTOURS),
    "metadata_source_contours": metadata_contours,
    "crs_cartes": "EPSG:4326",
    "crs_distances_surfaces": "EPSG:2154",
    "tolerance_diagnostic_metres": TOLERANCE_METRES,
    "simplification_cartes_metres": 20,
    "reaffectation_automatique_communes": False,
    "nombre_communes": len(profil_j11),
    "tables_controlees": manifeste_tables.to_dict(orient="records"),
    "versions": {
        "pandas": pd.__version__,
        "geopandas": gpd.__version__,
        "folium": folium.__version__,
    },
    "limites": [
        "Contrôle relatif aux contours utilisés.",
        "Position dans la commune distincte d'une adresse exacte.",
        "Seuil de proximité choisi pour le diagnostic.",
        "Absence de coordonnées conservée dans les comptages.",
        "Aucune réaffectation automatique des communes.",
        "Distances géométriques distinctes des distances piétonnes.",
        "Aucun score commercial calculé.",
    ],
}

with open(
    DOSSIER_INTERIM / "metadata_j11.json",
    "w",
    encoding="utf-8",
) as flux:
    json.dump(
        metadonnees,
        flux,
        ensure_ascii=False,
        indent=2,
    )

print("Contrôles structurels J11 terminés.")
print("Les anomalies géographiques restent à examiner selon le bilan.")

Contrôles structurels J11 terminés.
Les anomalies géographiques restent à examiner selon le bilan.
